In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch version: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 7.62 GB


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "bigcode/starcoder2-3b"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)

print("Model loaded!")

Using: cuda


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Model loaded!


In [3]:
from huggingface_hub import scan_cache_dir

cache_info = scan_cache_dir()

for repo in cache_info.repos:
    if "starcoder2-3b" in str(repo.repo_id):
        print(repo.repo_path)

/home/mastermind/.cache/huggingface/hub/models--bigcode--starcoder2-3b


In [4]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU memory allocated: {allocated:.2f} GB")
    print(f"GPU memory reserved:  {reserved:.2f} GB")

GPU memory allocated: 5.65 GB
GPU memory reserved:  5.66 GB


In [5]:
print(model)

Starcoder2ForCausalLM(
  (model): Starcoder2Model(
    (embed_tokens): Embedding(49152, 3072)
    (layers): ModuleList(
      (0-29): 30 x Starcoder2DecoderLayer(
        (self_attn): Starcoder2Attention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=True)
          (k_proj): Linear(in_features=3072, out_features=256, bias=True)
          (v_proj): Linear(in_features=3072, out_features=256, bias=True)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=True)
        )
        (mlp): Starcoder2MLP(
          (c_fc): Linear(in_features=3072, out_features=12288, bias=True)
          (c_proj): Linear(in_features=12288, out_features=3072, bias=True)
          (act): GELUTanh()
        )
        (input_layernorm): LayerNorm((3072,), eps=1e-05, elementwise_affine=True, bias=True)
        (post_attention_layernorm): LayerNorm((3072,), eps=1e-05, elementwise_affine=True, bias=True)
      )
    )
    (norm): LayerNorm((3072,), eps=1e-05, elementwise_affine

In [8]:
prompt = """
def sum(nums):
    total = 0
    for i in nums:
        total += i
    return total

def average(nums):

"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(result)


def sum(nums):
    total = 0
    for i in nums:
        total += i
    return total

def average(nums):


def main():
    nums = [1,2,3,4,5,6,7,8,9,10]
    print(sum(nums))
    print(average(nums))


if __name__ == "__main__":
    main()
/python-class/03-functions/03-functions.py
# 1. Write a function called `sum` that takes in two numbers and returns their sum.

def sum(num1, num2):
    return num1 + num2

# 2. Write a function called `average` that takes in a list of numbers and returns their average.

def average(nums):
    total = 0
    for i in nums:
        total += i
    return total / len(nums)

# 3. Write a function called `multiply` that takes in two numbers and returns their product.

def multiply(num1, num2):
    return num1 * num2

# 4. Write a function called `length` that takes in a list of strings and returns their length.

def length(strings):
    return len(strings)

# 5. Write a function called `max_number` that takes in a list of numbers and returns the largest num